In [7]:
import numpy as np
import pandas as pd

# Load the datasets created by data_analysis.ipynb.
processed_data = np.load("data/processed_datasets.npz")
X_train = pd.DataFrame(processed_data["X_train"])
X_val = pd.DataFrame(processed_data["X_val"])
X_test = pd.DataFrame(processed_data["X_test"])
Y_train = pd.Series(processed_data["Y_train"], name="churn")
Y_val = pd.Series(processed_data["Y_val"], name="churn")
Y_test = pd.Series(processed_data["Y_test"], name="churn")
processed_data.close()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("Y_train:", Y_train.shape)
print("Y_val:", Y_val.shape)
print("Y_test:", Y_test.shape)

X_train: (659028, 44)
X_val: (82379, 44)
X_test: (82379, 44)
Y_train: (659028,)
Y_val: (82379,)
Y_test: (82379,)


In [8]:
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from xgboost import XGBClassifier

In [9]:
# First XGBoost hyperparameter configuration.
FIRST_CONFIG = {
    "learning_rate": 0.03,
    "max_depth": 3,
    "n_estimators": 400,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}

SCALE_POS_WEIGHTS = [5, 7, 9, 12, 15]
DECISION_THRESHOLDS = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
N_FEATURES = 25

assert X_train.shape[1] >= N_FEATURES
print("Number of experiment runs:", len(SCALE_POS_WEIGHTS) * len(DECISION_THRESHOLDS))

Number of experiment runs: 45


In [10]:
def create_pipeline(scale_pos_weight):
    model = XGBClassifier(
        **FIRST_CONFIG,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    )
    return Pipeline([
        ("feature_selection", SelectKBest(score_func=f_classif, k=N_FEATURES)),
        ("model", model),
    ])


def evaluate_model(model, X, y, decision_threshold):
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= decision_threshold).astype(int)
    return {
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
    }

In [11]:
mlflow.set_tracking_uri("sqlite:///mlflow_churn.db")
mlflow.set_experiment("churn-xgboost-first-config-weight-threshold")

results = []

2026/09/29 11:44:58 INFO mlflow.tracking.fluent: Experiment with name 'churn-xgboost-first-config-weight-threshold' does not exist. Creating a new experiment.


In [12]:
for scale_pos_weight in SCALE_POS_WEIGHTS:
    for decision_threshold in DECISION_THRESHOLDS:
        run_name = (
            f"xgboost_weight_{scale_pos_weight}_"
            f"threshold_{decision_threshold:.2f}_top_{N_FEATURES}"
        )
        print("Running:", run_name, flush=True)

        pipeline = create_pipeline(scale_pos_weight)

        with mlflow.start_run(run_name=run_name):
            pipeline.fit(X_train, Y_train)
            metrics = evaluate_model(
                pipeline, X_val, Y_val, decision_threshold
            )

            mlflow.log_params({
                "model_type": "xgboost",
                "balance": "weighted",
                "n_features": N_FEATURES,
                "decision_threshold": decision_threshold,
                "scale_pos_weight": scale_pos_weight,
                **FIRST_CONFIG,
            })
            mlflow.log_metrics(metrics)
            mlflow.sklearn.log_model(
                pipeline, name="model", serialization_format="cloudpickle"
            )

            results.append({
                "model": "xgboost",
                "balance": "weighted",
                "n_features": N_FEATURES,
                "scale_pos_weight": scale_pos_weight,
                "decision_threshold": decision_threshold,
                **FIRST_CONFIG,
                **metrics,
            })
            print(
                "Finished:", run_name,
                "F1:", round(metrics["f1"], 4),
                flush=True,
            )

Running: xgboost_weight_5_threshold_0.20_top_25


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 11:45:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: xgboost_weight_5_threshold_0.20_top_25 F1: 0.2048
Running: xgboost_weight_5_threshold_0.25_top_25


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 11:45:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: xgboost_weight_5_threshold_0.25_top_25 F1: 0.2199
Running: xgboost_weight_5_threshold_0.30_top_25


...

Finished: xgboost_weight_15_threshold_0.50_top_25 F1: 0.2199
Running: xgboost_weight_15_threshold_0.55_top_25


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 11:55:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: xgboost_weight_15_threshold_0.55_top_25 F1: 0.2328
Running: xgboost_weight_15_threshold_0.60_top_25


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 11:55:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: xgboost_weight_15_threshold_0.60_top_25 F1: 0.2486


In [13]:
results_df = pd.DataFrame(results).sort_values(
    by=["f1", "recall", "precision"], ascending=False
).reset_index(drop=True)
results_df.to_csv("data/xgboost_weight_threshold_results.csv", index=False)
results_df

,model,balance,n_features,scale_pos_weight,decision_threshold,learning_rate,max_depth,n_estimators,subsample,colsample_bytree,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,xgboost,weighted,25,7,0.50,0.03,3,400,0.8,0.8,0.736183,0.625042,0.184696,0.486410,0.267731,0.684104,0.206652
1,xgboost,weighted,25,5,0.45,0.03,3,400,0.8,0.8,0.781643,0.612140,0.199988,0.400710,0.266813,0.684107,0.206486
2,xgboost,weighted,25,9,0.60,0.03,3,400,0.8,0.8,0.787846,0.609918,0.202531,0.387977,0.266135,0.684124,0.206624
3,xgboost,weighted,25,7,0.55,0.03,3,400,0.8,0.8,0.801260,0.605977,0.209572,0.362390,0.265566,0.684104,0.206652
4,xgboost,weighted,25,9,0.55,0.03,3,400,0.8,0.8,0.716518,0.627529,0.178597,0.516528,0.265421,0.684124,0.206624
5,xgboost,weighted,25,5,0.40,0.03,3,400,0.8,0.8,0.708397,0.627870,0.176052,0.527424,0.263987,0.684107,0.206486
6,xgboost,weighted,25,12,0.60,0.03,3,400,0.8,0.8,0.681630,0.630446,0.169431,0.566601,0.260857,0.684150,0.206562
7,xgboost,weighted,25,7,0.45,0.03,3,400,0.8,0.8,0.652278,0.632513,0.163285,0.607860,0.257421,0.684104,0.206652
8,xgboost,weighted,25,9,0.50,0.03,3,400,0.8,0.8,0.630209,0.633775,0.159317,0.638222,0.254983,0.684124,0.206624
9,xgboost,weighted,25,5,0.50,0.03,3,400,0.8,0.8,0.833513,0.589558,0.228275,0.285260,0.253605,0.684107,0.206486


In [14]:
best_result = results_df.iloc[0]
best_result

model                  xgboost
balance               weighted
n_features                  25
scale_pos_weight             7
decision_threshold         0.5
learning_rate             0.03
max_depth                    3
n_estimators               400
subsample                  0.8
colsample_bytree           0.8
accuracy              0.736183
balanced_accuracy     0.625042
precision             0.184696
recall                 0.48641
f1                    0.267731
roc_auc               0.684104
average_precision     0.206652
Name: 0, dtype: object

## Held-out test evaluation
Evaluate the saved model selected by validation F1 (then recall and precision). Keep the selected threshold fixed; do not tune on test results. The following cell runs independently without repeating training.

In [1]:
# This cell can run independently of the training sweep above.
import sqlite3
from pathlib import Path

import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score,
)

# Freeze the configuration selected using validation data only.
validation_results = pd.read_csv("data/xgboost_weight_threshold_results.csv")
selected = validation_results.sort_values(
    ["f1", "recall", "precision"], ascending=False
).iloc[0]
threshold = float(selected["decision_threshold"])
run_name = (
    f"xgboost_weight_{int(selected['scale_pos_weight'])}_"
    f"threshold_{threshold:.2f}_top_{int(selected['n_features'])}"
)
# Read local tracking metadata without changing the training runs.
with sqlite3.connect("file:mlflow_churn.db?mode=ro", uri=True) as connection:
    saved_models = connection.execute(
        """SELECT r.run_uuid, m.artifact_location
           FROM runs r JOIN experiments e ON e.experiment_id = r.experiment_id
           JOIN logged_models m ON m.source_run_id = r.run_uuid
           WHERE e.name = ? AND r.name = ? AND r.status = 'FINISHED'
             AND r.lifecycle_stage = 'active' AND m.lifecycle_stage = 'active'
           ORDER BY r.start_time DESC""",
        ("churn-xgboost-first-config-weight-threshold", run_name),
    ).fetchall()
if len(saved_models) != 1:
    raise ValueError(f"Expected one saved model for {run_name}; found {len(saved_models)}")
source_run_id, model_path = saved_models[0]
test_pipeline = mlflow.sklearn.load_model(model_path)
assert test_pipeline.named_steps["model"].get_params()["scale_pos_weight"] == selected["scale_pos_weight"]

# Check that the saved pipeline reproduces the recorded validation score.
with np.load("data/processed_datasets.npz") as datasets:
    validation_probabilities = test_pipeline.predict_proba(pd.DataFrame(datasets["X_val"]))[:, 1]
    validation_f1 = f1_score(datasets["Y_val"], validation_probabilities >= threshold)
    if not np.isclose(validation_f1, selected["f1"], atol=1e-10, rtol=0):
        raise ValueError("Saved model/data do not reproduce the selected validation F1")
    test_X = pd.DataFrame(datasets["X_test"])
    test_y = datasets["Y_test"]

probabilities = test_pipeline.predict_proba(test_X)[:, 1]
predictions = (probabilities >= threshold).astype(int)
test_metrics = {
    "accuracy": accuracy_score(test_y, predictions),
    "balanced_accuracy": balanced_accuracy_score(test_y, predictions),
    "precision": precision_score(test_y, predictions, zero_division=0),
    "recall": recall_score(test_y, predictions, zero_division=0),
    "f1": f1_score(test_y, predictions, zero_division=0),
    "roc_auc": roc_auc_score(test_y, probabilities),
    "average_precision": average_precision_score(test_y, probabilities),
}
comparison = pd.DataFrame({
    "validation": {metric: float(selected[metric]) for metric in test_metrics},
    "test": test_metrics,
})
confusion = pd.DataFrame(
    confusion_matrix(test_y, predictions, labels=[0, 1]),
    index=["actual_no_churn", "actual_churn"],
    columns=["predicted_no_churn", "predicted_churn"],
)
test_results = pd.DataFrame([{
    "source_run_id": source_run_id,
    "scale_pos_weight": int(selected["scale_pos_weight"]),
    "decision_threshold": threshold,
    "n_test": len(test_y),
    **test_metrics,
}])
test_results.to_csv("data/xgboost_weighted_test_results.csv", index=False)
confusion.to_csv("data/xgboost_weighted_test_confusion_matrix.csv")
print(f"Source run: {source_run_id}")
print(f"Weight: {selected['scale_pos_weight']}; threshold: {threshold}; test rows: {len(test_y)}")
print(comparison.round(6).to_string())
print("\nConfusion matrix (rows = actual, columns = predicted):")
print(confusion.to_string())
print("\nClassification report:")
print(classification_report(test_y, predictions, labels=[0, 1],
                            target_names=["no_churn", "churn"], zero_division=0, digits=4))


Source run: 79b29732a30247d2a9666decd4f88646
Weight: 7; threshold: 0.5; test rows: 82379
                   validation      test
accuracy             0.736183  0.741706
balanced_accuracy    0.625042  0.632139
precision            0.184696  0.190860
recall               0.486410  0.495470
f1                   0.267731  0.275569
roc_auc              0.684104  0.693321
average_precision    0.206652  0.211564

Confusion matrix (rows = actual, columns = predicted):
                 predicted_no_churn  predicted_churn
actual_no_churn               57054            17157
actual_churn                   4121             4047

Classification report:
              precision    recall  f1-score   support

    no_churn     0.9326    0.7688    0.8428     74211
       churn     0.1909    0.4955    0.2756      8168

    accuracy                         0.7417     82379
   macro avg     0.5617    0.6321    0.5592     82379
weighted avg     0.8591    0.7417    0.7866     82379

